In [1]:
import os
import json
from dotenv import load_dotenv

In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.callbacks.manager import get_openai_callback

In [3]:
load_dotenv()
key = os.getenv("GROQ_API_KEY")

In [4]:
openaiModel =  ChatGroq(model="openai/gpt-oss-120b",temperature = 0.3)

In [5]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [30]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [31]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [ ]:
TEMPLATE1="""
You are an expert in {subject}. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
Quiz_MCQs:
{quiz}

Check from an subject expert for the above quiz:
"""

In [42]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE1)

In [43]:
quiz_chain = (
    quiz_generation_prompt
    | openaiModel
    | StrOutputParser()
)

In [44]:
review_chain = (
    quiz_evaluation_prompt
    | openaiModel
    | StrOutputParser()
)

In [45]:
generate_evaluate_chain = (
    RunnablePassthrough.assign(
        quiz=quiz_chain
    )
    .assign(
        review=(
            lambda x: {
                "quiz": x["quiz"],
                "subject": x["subject"],
            }
        )
        | review_chain
    )
)

In [46]:
from pathlib import Path
file_path=r"D:\Projects\mcqgen\experiment\test.txt"

In [47]:
file_path

'D:\\Projects\\mcqgen\\experiment\\test.txt'

In [48]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [49]:
print(TEXT)

Long Short-Term Memory (LSTM) is a specialized type of Recurrent Neural Network (RNN) designed to process sequential data, such as time-series forecasting, speech recognition, and language translation. Unlike traditional networks, LSTMs are engineered to remember vital information over long periods, making them incredibly effective at understanding context and predicting future steps based on past inputs.The secret to an LSTM's power lies in its unique internal structure, which replaces the repeating modules of standard RNNs with a cell state and a series of "gates". The cell state acts as a conveyor belt, carrying information straight through the entire chain of processing. The gatesâ€”specifically the forget gate, input gate, and output gateâ€”are small neural networks that act as valves. They determine exactly which pieces of information are allowed onto the conveyor belt to be stored, which are updated, and which are discarded as noise.These gating mechanisms allow LSTMs to solve t

In [50]:
NUMBER=5 
SUBJECT="Deep Learning"
TONE="simple"

In [51]:
#setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain.invoke(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )

In [52]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:2543
Prompt Tokens:1106
Completion Tokens:1437
Total Cost:0.0


In [53]:
response

{'text': 'Long Short-Term Memory (LSTM) is a specialized type of Recurrent Neural Network (RNN) designed to process sequential data, such as time-series forecasting, speech recognition, and language translation. Unlike traditional networks, LSTMs are engineered to remember vital information over long periods, making them incredibly effective at understanding context and predicting future steps based on past inputs.The secret to an LSTM\'s power lies in its unique internal structure, which replaces the repeating modules of standard RNNs with a cell state and a series of "gates". The cell state acts as a conveyor belt, carrying information straight through the entire chain of processing. The gatesâ€”specifically the forget gate, input gate, and output gateâ€”are small neural networks that act as valves. They determine exactly which pieces of information are allowed onto the conveyor belt to be stored, which are updated, and which are discarded as noise.These gating mechanisms allow LSTMs